env test

In [1]:
import sys
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ModuleNotFoundError: No module named 'torch'

In [ ]:
! pip install -U ultralytics

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n-pose.pt")

print("YOLO11n-pose loaded successfully.")
model.info()

In [ ]:
source="D:\Documents\2026-04\Software\yoga train"

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n-pose.pt")

image_paths = [
    "D:/Documents/2026-04/Software/yoga train/squat.jpg",
    "D:/Documents/2026-04/Software/yoga train/Downward-Dog.jpg"
]

results = model.predict(
    source=image_paths,
    conf=0.3,
    device=0,
    save=True
)

print("Prediction finished.")
print("Saved to:", results[0].save_dir)





In [ ]:
import os
from IPython.display import Image, display

save_dir = results[0].save_dir
print("Saved directory:", save_dir)

for file in os.listdir(save_dir):
    if file.lower().endswith((".jpg", ".jpeg", ".png")):
        print(file)
        display(Image(filename=os.path.join(save_dir, file)))

In [ ]:
keypoint_names = [
    "nose",
    "left_eye", "right_eye",
    "left_ear", "right_ear",
    "left_shoulder", "right_shoulder",
    "left_elbow", "right_elbow",
    "left_wrist", "right_wrist",
    "left_hip", "right_hip",
    "left_knee", "right_knee",
    "left_ankle", "right_ankle"
]

for img_path, r in zip(image_paths, results):
    print("=" * 80)
    print("Image:", img_path)

    if r.keypoints is None or len(r.keypoints) == 0:
        print("No keypoints detected.")
        continue

    xy = r.keypoints.xy[0].cpu().numpy()
    conf = r.keypoints.conf[0].cpu().numpy()

    for i, name in enumerate(keypoint_names):
        x, y = xy[i]
        c = conf[i]
        print(f"{i:2d} {name:15s} x={x:8.1f}, y={y:8.1f}, conf={c:.3f}")

compare with yolo11s:

In [ ]:
from ultralytics import YOLO

model_s = YOLO("yolo11s-pose.pt")

results_s = model_s.predict(
    source=image_paths,
    conf=0.3,
    device=0,
    save=True,
    project="D:/Documents/2026-04/Software/yoga train/runs",
    name="yolo11s_test",
    exist_ok=True
)

print("YOLO11s prediction finished.")
print("Saved to:", results_s[0].save_dir)

In [ ]:
import os
from IPython.display import Image, display

save_dir_s = results_s[0].save_dir
print("Saved directory:", save_dir_s)

for file in os.listdir(save_dir_s):
    if file.lower().endswith((".jpg", ".jpeg", ".png")):
        print(file)
        display(Image(filename=os.path.join(save_dir_s, file)))

In [ ]:
for img_path, r in zip(image_paths, results_s):
    print("=" * 80)
    print("Image:", img_path)

    if r.keypoints is None or len(r.keypoints) == 0:
        print("No keypoints detected.")
        continue

    xy = r.keypoints.xy[0].cpu().numpy()
    conf = r.keypoints.conf[0].cpu().numpy()

    for i, name in enumerate(keypoint_names):
        x, y = xy[i]
        c = conf[i]
        print(f"{i:2d} {name:15s} x={x:8.1f}, y={y:8.1f}, conf={c:.3f}")

test running time

In [ ]:
import time
from ultralytics import YOLO

model = YOLO("yolo11s-pose.pt")

image_paths = [
    "D:/Documents/2026-04/Software/yoga train/squat.jpg",
    "D:/Documents/2026-04/Software/yoga train/Downward-Dog.jpg"
]

# 预热 GPU
for _ in range(5):
    _ = model.predict(
        source=image_paths[0],
        conf=0.3,
        device=0,
        verbose=False
    )

# 正式测速
times = []

for i in range(30):
    start = time.time()

    _ = model.predict(
        source=image_paths,
        conf=0.3,
        device=0,
        verbose=False
    )

    end = time.time()
    times.append((end - start) * 1000)

avg_time = sum(times) / len(times)

print(f"Average inference time for 2 images: {avg_time:.2f} ms")
print(f"Average inference time per image: {avg_time / len(image_paths):.2f} ms")

we choose yolo11s for better performance, and the following is Fine-Tuning

1. download dataset roboflow

find suitable dataset here:

https://app.roboflow.com/jianxing-song/yoga-pose-uq4bq-zlayz-qowrn/browse?queryText=&pageSize=50&startingIndex=0&browseQuery=true

In [ ]:
from pathlib import Path

DATASET_DIR = r"D:/Documents/2026-04/Software/yoga train/Yoga Pose.yolov8"

root = Path(DATASET_DIR)

print("Dataset exists:", root.exists())
print("Dataset root:", root)

print("\nTop-level files/folders:")
if root.exists():
    for p in root.iterdir():
        print(" -", p.name)

print("\nSearching for yaml files:")
yaml_files = list(root.rglob("*.yaml"))
for yaml_file in yaml_files:
    print(" -", yaml_file)

print("\nImage/label folder counts:")
for split in ["train", "valid", "val", "test"]:
    img_dir = root / split / "images"
    label_dir = root / split / "labels"

    if img_dir.exists():
        imgs = list(img_dir.glob("*"))
        print(f"{split}/images:", len(imgs))

    if label_dir.exists():
        labels = list(label_dir.glob("*.txt"))
        print(f"{split}/labels:", len(labels))

In [ ]:
from pathlib import Path

yaml_files = list(Path(DATASET_DIR).rglob("*.yaml"))

for y in yaml_files:
    print("=" * 80)
    print(y)
    print(y.read_text(encoding="utf-8"))

Test dataset

In [ ]:
from pathlib import Path
import random

DATASET_DIR = r"D:/Documents/2026-04/Software/yoga train/Yoga Pose.yolov8"
root = Path(DATASET_DIR)

label_dir = root / "train" / "labels"
label_files = list(label_dir.glob("*.txt"))

print("Number of label files:", len(label_files))

sample_files = random.sample(label_files, min(10, len(label_files)))

for lf in sample_files:
    print("=" * 80)
    print("Label file:", lf.name)

    text = lf.read_text(encoding="utf-8").strip()

    if not text:
        print("Empty label file")
        continue

    lines = text.splitlines()
    print("Number of objects:", len(lines))

    for i, line in enumerate(lines[:3]):
        parts = line.strip().split()
        print(f"Line {i}: number of values =", len(parts))
        print(line[:300])

try coco

In [ ]:
from pathlib import Path
import json

COCO_DIR = Path(r"D:/Documents/2026-04/Software/yoga train/Yoga Pose.coco")

print("COCO dir exists:", COCO_DIR.exists())
print("COCO root:", COCO_DIR)

print("\nTop-level files/folders:")
for p in COCO_DIR.iterdir():
    print(" -", p.name)

print("\nSearching for json files:")
json_files = list(COCO_DIR.rglob("*.json"))
for jf in json_files:
    print(" -", jf)

print("\nNumber of json files:", len(json_files))

In [ ]:
import json
from pathlib import Path

json_files = list(Path(r"D:/Documents/2026-04/Software/yoga train/Yoga Pose.coco").rglob("*.json"))

for jf in json_files:
    print("=" * 100)
    print("JSON:", jf)

    data = json.loads(jf.read_text(encoding="utf-8"))

    print("Top-level keys:", data.keys())

    annotations = data.get("annotations", [])
    categories = data.get("categories", [])

    print("Number of annotations:", len(annotations))
    print("Number of categories:", len(categories))

    if categories:
        print("\nCategories sample:")
        for cat in categories[:5]:
            print(cat)

    if annotations:
        print("\nFirst annotation keys:")
        print(annotations[0].keys())

        print("\nFirst annotation:")
        print(annotations[0])

        has_keypoints = any("keypoints" in ann for ann in annotations)
        print("\nHas keypoints field:", has_keypoints)

        if has_keypoints:
            valid_kpt_count = 0
            for ann in annotations:
                if "keypoints" in ann and len(ann["keypoints"]) > 0:
                    valid_kpt_count += 1

            print("Annotations with non-empty keypoints:", valid_kpt_count)

            first_kpt_ann = next(
                ann for ann in annotations
                if "keypoints" in ann and len(ann["keypoints"]) > 0
            )

            print("Length of keypoints:", len(first_kpt_ann["keypoints"]))
            print("num_keypoints:", first_kpt_ann.get("num_keypoints"))

In [ ]:
from pathlib import Path
import json

COCO_DIR = Path(r"D:/Documents/2026-04/Software/yoga train/Yoga Pose.coco")

json_files = list(COCO_DIR.rglob("*.json"))
print("JSON files found:")
for jf in json_files:
    print(" -", jf)

print("\n" + "=" * 100)

# 只看第一个 json 就够了
jf = json_files[0]
data = json.loads(jf.read_text(encoding="utf-8"))

print("Categories full content:\n")
for i, cat in enumerate(data["categories"]):
    print(f"[Category {i}]")
    for k, v in cat.items():
        print(f"{k}: {v}")
    print("-" * 80)

In [ ]:
from pathlib import Path
import json
import cv2
import matplotlib.pyplot as plt

COCO_DIR = Path(r"D:/Documents/2026-04/Software/yoga train/Yoga Pose.coco")
json_path = COCO_DIR / "train" / "_annotations.coco.json"
image_dir = COCO_DIR / "train"

data = json.loads(json_path.read_text(encoding="utf-8"))

# category id 映射
cat_id_to_name = {cat["id"]: cat["name"] for cat in data["categories"]}
image_id_to_info = {img["id"]: img for img in data["images"]}

print("Categories:")
for k, v in cat_id_to_name.items():
    print(k, v)

# 找一张 Downdog 图片
target_class_name = "Downdog"
target_cat_id = None

for cat_id, name in cat_id_to_name.items():
    if name == target_class_name:
        target_cat_id = cat_id
        break

print("Target category id:", target_cat_id)

target_ann = None
for ann in data["annotations"]:
    if ann["category_id"] == target_cat_id and "keypoints" in ann and len(ann["keypoints"]) > 0:
        target_ann = ann
        break

if target_ann is None:
    raise ValueError("No annotation found for target class.")

img_info = image_id_to_info[target_ann["image_id"]]
img_path = image_dir / img_info["file_name"]

print("Image:", img_path)
print("Annotation category:", cat_id_to_name[target_ann["category_id"]])
print("Keypoints length:", len(target_ann["keypoints"]))

# 读图
img = cv2.imread(str(img_path))
if img is None:
    raise FileNotFoundError(img_path)

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 解析关键点
kpts = target_ann["keypoints"]

plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)

for i in range(0, len(kpts), 3):
    idx = i // 3
    x, y, v = kpts[i], kpts[i + 1], kpts[i + 2]

    if v > 0:
        plt.scatter(x, y, s=50)
        plt.text(x + 5, y + 5, str(idx), fontsize=14)

plt.title(f"{target_class_name} keypoint index visualization")
plt.axis("off")
plt.show()

print("\nKeypoint values:")
for i in range(0, len(kpts), 3):
    idx = i // 3
    x, y, v = kpts[i], kpts[i + 1], kpts[i + 2]
    print(f"{idx:2d}: x={x:8.2f}, y={y:8.2f}, v={v}")

check keypoint length for every category

In [ ]:
from pathlib import Path
import json
from collections import defaultdict, Counter

COCO_DIR = Path(r"D:/Documents/2026-04/Software/yoga train/Yoga Pose.coco")
json_path = COCO_DIR / "train" / "_annotations.coco.json"

data = json.loads(json_path.read_text(encoding="utf-8"))

cat_id_to_name = {cat["id"]: cat["name"] for cat in data["categories"]}

length_stats = defaultdict(Counter)
empty_count = defaultdict(int)
total_count = defaultdict(int)

for ann in data["annotations"]:
    cat_id = ann["category_id"]
    cat_name = cat_id_to_name.get(cat_id, str(cat_id))
    total_count[cat_name] += 1

    kpts = ann.get("keypoints", [])

    if not kpts:
        empty_count[cat_name] += 1
    else:
        length_stats[cat_name][len(kpts)] += 1

print("Keypoint length statistics by category:")
print("=" * 80)

for cat_name in sorted(total_count.keys()):
    print(f"\nCategory: {cat_name}")
    print("Total annotations:", total_count[cat_name])
    print("Empty keypoints:", empty_count[cat_name])
    print("Keypoint lengths:")

    for length, count in sorted(length_stats[cat_name].items()):
        print(f"  length={length:3d}, points={length // 3:2d}, count={count}")

the keypoint is not same, we need to adjust the dataset

In [ ]:
from pathlib import Path
import json
import shutil
from collections import defaultdict

COCO_DIR = Path(r"D:/Documents/2026-04/Software/yoga train/Yoga Pose.coco")
json_path = COCO_DIR / "train" / "_annotations.coco.json"
image_dir = COCO_DIR / "train"

OUT_DIR = Path(r"D:/Documents/2026-04/Software/yoga train/relabel_images")
OUT_DIR.mkdir(parents=True, exist_ok=True)

data = json.loads(json_path.read_text(encoding="utf-8"))

cat_id_to_name = {cat["id"]: cat["name"] for cat in data["categories"]}
image_id_to_info = {img["id"]: img for img in data["images"]}

target_classes = {"Downdog", "Plank"}

selected = defaultdict(list)

for ann in data["annotations"]:
    cat_name = cat_id_to_name.get(ann["category_id"])

    if cat_name not in target_classes:
        continue

    img_info = image_id_to_info[ann["image_id"]]
    img_name = img_info["file_name"]

    src = image_dir / img_name

    if not src.exists():
        continue

    class_out_dir = OUT_DIR / cat_name
    class_out_dir.mkdir(parents=True, exist_ok=True)

    dst = class_out_dir / img_name

    if not dst.exists():
        shutil.copy2(src, dst)

    selected[cat_name].append(img_name)

print("Copied images:")
for cls, imgs in selected.items():
    unique_count = len(set(imgs))
    print(f"{cls}: {unique_count} images")

print("Output directory:", OUT_DIR)

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import random
import cv2
import csv
from collections import Counter

# =========================
# 路径配置
# =========================
BASE_DIR = Path(r"D:/Documents/2026-04/Software/yoga train")
IMG_ROOT = BASE_DIR / "relabel_images"

OUT_ROOT = BASE_DIR / "auto_prelabel_preview"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

CSV_PATH = OUT_ROOT / "auto_prelabel_summary.csv"

# 每个类别先抽取一部分做测试，不要一开始全跑
SAMPLE_PER_CLASS = {
    "Downdog": 30,
    "Plank": 30,
}

# =========================
# 加载模型
# =========================
model = YOLO("yolo11m-pose.pt")

# COCO 17 keypoints index
KEYPOINT_NAMES = [
    "nose",
    "left_eye", "right_eye",
    "left_ear", "right_ear",
    "left_shoulder", "right_shoulder",
    "left_elbow", "right_elbow",
    "left_wrist", "right_wrist",
    "left_hip", "right_hip",
    "left_knee", "right_knee",
    "left_ankle", "right_ankle"
]

# 对瑜伽动作比较重要的点：肩、肘、腕、髋、膝、踝
REQUIRED_KPTS = [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]

def evaluate_result(result, kpt_threshold=0.5):
    """
    粗略评估自动预测质量。
    只是用来筛选，不作为最终标准。
    """
    if result.boxes is None or len(result.boxes) == 0:
        return "no_person", 0, 0, 0.0

    num_person = len(result.boxes)

    # 取 person 置信度最高的那个检测结果
    best_idx = int(result.boxes.conf.argmax().item())
    person_conf = float(result.boxes.conf[best_idx].item())

    if result.keypoints is None or result.keypoints.conf is None:
        return "no_keypoints", num_person, 0, person_conf

    kpt_conf = result.keypoints.conf[best_idx].cpu().numpy()
    reliable_count = sum(kpt_conf[i] >= kpt_threshold for i in REQUIRED_KPTS)

    if num_person > 1:
        status = "multi_person_or_split"
    elif reliable_count >= 10:
        status = "good_candidate"
    elif reliable_count >= 7:
        status = "need_review"
    else:
        status = "bad_candidate"

    return status, num_person, reliable_count, person_conf


# =========================
# 随机抽样并预测
# =========================
random.seed(42)

rows = []
status_counter = Counter()

for cls_name, sample_num in SAMPLE_PER_CLASS.items():
    cls_dir = IMG_ROOT / cls_name
    image_files = []

    for ext in ["*.jpg", "*.jpeg", "*.png"]:
        image_files.extend(list(cls_dir.glob(ext)))

    image_files = sorted(image_files)

    if len(image_files) == 0:
        print(f"No images found for {cls_name}")
        continue

    sample_files = random.sample(image_files, min(sample_num, len(image_files)))

    out_cls_dir = OUT_ROOT / cls_name
    out_cls_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nProcessing {cls_name}: {len(sample_files)} images")

    for img_path in sample_files:
        result = model.predict(
            source=str(img_path),
            conf=0.25,
            imgsz=640,
            device=0,
            verbose=False
        )[0]

        status, num_person, reliable_count, person_conf = evaluate_result(result)

        status_counter[(cls_name, status)] += 1

        # 保存可视化预测图
        plotted = result.plot()
        out_img_path = out_cls_dir / f"{img_path.stem}__{status}.jpg"
        cv2.imwrite(str(out_img_path), plotted)

        rows.append({
            "class": cls_name,
            "image": img_path.name,
            "status": status,
            "num_person": num_person,
            "reliable_required_keypoints": reliable_count,
            "person_conf": round(person_conf, 4),
            "preview_path": str(out_img_path)
        })

# =========================
# 保存 CSV 统计
# =========================
with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "class",
            "image",
            "status",
            "num_person",
            "reliable_required_keypoints",
            "person_conf",
            "preview_path"
        ]
    )
    writer.writeheader()
    writer.writerows(rows)

print("\nDone.")
print("Preview images saved to:", OUT_ROOT)
print("Summary CSV:", CSV_PATH)

print("\nStatus summary:")
for (cls_name, status), count in sorted(status_counter.items()):
    print(f"{cls_name:10s} | {status:22s} | {count}")

generate new dataset, prepare for next step

In [ ]:
from pathlib import Path
import json
import shutil
from collections import defaultdict

# =========================
# 路径配置
# =========================
BASE_DIR = Path(r"D:/Documents/2026-04/Software/yoga train")

COCO_DIR = BASE_DIR / "Yoga Pose.coco"
json_path = COCO_DIR / "train" / "_annotations.coco.json"
image_dir = COCO_DIR / "train"

OUT_DIR = BASE_DIR / "relabel_images_all"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# 读取 COCO 标注
# =========================
data = json.loads(json_path.read_text(encoding="utf-8"))

cat_id_to_name = {cat["id"]: cat["name"] for cat in data["categories"]}
image_id_to_info = {img["id"]: img for img in data["images"]}

target_classes = {"Downdog", "Plank", "Tree", "Goddess", "Warrior2"}

selected = defaultdict(set)

# =========================
# 复制图片
# =========================
for ann in data["annotations"]:
    cat_name = cat_id_to_name.get(ann["category_id"])

    if cat_name not in target_classes:
        continue

    img_info = image_id_to_info[ann["image_id"]]
    img_name = img_info["file_name"]

    src = image_dir / img_name

    if not src.exists():
        print("Missing image:", src)
        continue

    class_out_dir = OUT_DIR / cat_name
    class_out_dir.mkdir(parents=True, exist_ok=True)

    dst = class_out_dir / img_name

    if not dst.exists():
        shutil.copy2(src, dst)

    selected[cat_name].add(img_name)

# =========================
# 打印结果
# =========================
print("Copied images:")
for cls in sorted(target_classes):
    print(f"{cls}: {len(selected[cls])} images")

print("\nOutput directory:")
print(OUT_DIR)

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import cv2
import shutil
import csv
from collections import Counter

# =========================
# 路径配置
# =========================
BASE_DIR = Path(r"D:/Documents/2026-04/Software/yoga train")

IMG_ROOT = BASE_DIR / "relabel_images_all"

OUT_ROOT = BASE_DIR / "pseudo_yolo_pose_dataset"
IMG_OUT = OUT_ROOT / "images" / "train"
LABEL_OUT = OUT_ROOT / "labels" / "train"
PREVIEW_OUT = OUT_ROOT / "previews" / "train"

CSV_PATH = OUT_ROOT / "pseudo_label_summary.csv"

# 如果之前运行过，建议清空旧输出，避免混入旧文件
CLEAN_OUTPUT = True

if CLEAN_OUTPUT and OUT_ROOT.exists():
    shutil.rmtree(OUT_ROOT)

for p in [IMG_OUT, LABEL_OUT, PREVIEW_OUT]:
    p.mkdir(parents=True, exist_ok=True)

# =========================
# 模型配置
# =========================
teacher_model = YOLO("yolo11m-pose.pt")

PERSON_CONF_THRES = 0.25
KPT_CONF_THRES = 0.30

NUM_KPTS = 17
CLASS_ID = 0

CLASSES = ["Downdog", "Plank", "Tree", "Goddess", "Warrior2"]

# COCO 17关键点顺序：
# 0 nose
# 1 left_eye
# 2 right_eye
# 3 left_ear
# 4 right_ear
# 5 left_shoulder
# 6 right_shoulder
# 7 left_elbow
# 8 right_elbow
# 9 left_wrist
# 10 right_wrist
# 11 left_hip
# 12 right_hip
# 13 left_knee
# 14 right_knee
# 15 left_ankle
# 16 right_ankle


def normalize_bbox_xyxy(box_xyxy, img_w, img_h):
    x1, y1, x2, y2 = box_xyxy

    x1 = max(0, min(float(x1), img_w - 1))
    y1 = max(0, min(float(y1), img_h - 1))
    x2 = max(0, min(float(x2), img_w - 1))
    y2 = max(0, min(float(y2), img_h - 1))

    xc = ((x1 + x2) / 2) / img_w
    yc = ((y1 + y2) / 2) / img_h
    bw = (x2 - x1) / img_w
    bh = (y2 - y1) / img_h

    return xc, yc, bw, bh


def make_yolo_pose_label(result, img_w, img_h):
    """
    从 YOLO pose result 生成一行 YOLO pose label。
    只保留置信度最高的 person。
    """
    if result.boxes is None or len(result.boxes) == 0:
        return None, "no_person", 0, 0, 0.0

    num_person = len(result.boxes)

    best_idx = int(result.boxes.conf.argmax().item())
    person_conf = float(result.boxes.conf[best_idx].item())

    if person_conf < PERSON_CONF_THRES:
        return None, "low_person_conf", num_person, 0, person_conf

    if result.keypoints is None or result.keypoints.xy is None or result.keypoints.conf is None:
        return None, "no_keypoints", num_person, 0, person_conf

    box_xyxy = result.boxes.xyxy[best_idx].cpu().numpy()
    xc, yc, bw, bh = normalize_bbox_xyxy(box_xyxy, img_w, img_h)

    kpt_xy = result.keypoints.xy[best_idx].cpu().numpy()
    kpt_conf = result.keypoints.conf[best_idx].cpu().numpy()

    label_values = [
        str(CLASS_ID),
        f"{xc:.6f}",
        f"{yc:.6f}",
        f"{bw:.6f}",
        f"{bh:.6f}",
    ]

    visible_count = 0

    for i in range(NUM_KPTS):
        x, y = kpt_xy[i]
        c = kpt_conf[i]

        if c >= KPT_CONF_THRES:
            x_norm = max(0.0, min(float(x) / img_w, 1.0))
            y_norm = max(0.0, min(float(y) / img_h, 1.0))
            v = 2
            visible_count += 1
        else:
            x_norm = 0.0
            y_norm = 0.0
            v = 0

        label_values.extend([
            f"{x_norm:.6f}",
            f"{y_norm:.6f}",
            str(v)
        ])

    if num_person > 1:
        status = "multi_person_or_split"
    elif visible_count >= 12:
        status = "good_candidate"
    elif visible_count >= 8:
        status = "need_review"
    else:
        status = "bad_candidate"

    return " ".join(label_values), status, num_person, visible_count, person_conf


# =========================
# 收集图片
# =========================
image_items = []

for cls_name in CLASSES:
    cls_dir = IMG_ROOT / cls_name

    if not cls_dir.exists():
        print(f"Warning: missing folder {cls_dir}")
        continue

    for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]:
        for img_path in cls_dir.glob(ext):
            image_items.append((cls_name, img_path))

print("Total images found:", len(image_items))

if len(image_items) == 0:
    raise RuntimeError("No images found. Please check IMG_ROOT path.")


# =========================
# 自动标注
# =========================
rows = []
counter = Counter()

for idx, (cls_name, img_path) in enumerate(image_items, start=1):
    if idx % 50 == 0 or idx == 1:
        print(f"Processing {idx}/{len(image_items)}")

    img = cv2.imread(str(img_path))
    if img is None:
        counter["read_failed"] += 1
        rows.append({
            "class": cls_name,
            "image": img_path.name,
            "status": "read_failed",
            "num_person": 0,
            "visible_keypoints": 0,
            "person_conf": 0,
            "image_out": "",
            "label_out": "",
            "preview_out": ""
        })
        continue

    img_h, img_w = img.shape[:2]

    result = teacher_model.predict(
        source=str(img_path),
        conf=PERSON_CONF_THRES,
        imgsz=640,
        device=0,
        verbose=False
    )[0]

    label_line, status, num_person, visible_count, person_conf = make_yolo_pose_label(
        result,
        img_w,
        img_h
    )

    counter[status] += 1

    safe_name = f"{cls_name}_{img_path.stem}"
    suffix = img_path.suffix.lower()

    image_out_path = IMG_OUT / f"{safe_name}{suffix}"
    label_out_path = LABEL_OUT / f"{safe_name}.txt"
    preview_out_path = PREVIEW_OUT / f"{safe_name}__{status}.jpg"

    shutil.copy2(img_path, image_out_path)

    if label_line is not None:
        label_out_path.write_text(label_line + "\n", encoding="utf-8")
    else:
        label_out_path.write_text("", encoding="utf-8")

    try:
        plotted = result.plot()
        cv2.imwrite(str(preview_out_path), plotted)
    except Exception:
        cv2.imwrite(str(preview_out_path), img)

    rows.append({
        "class": cls_name,
        "image": img_path.name,
        "status": status,
        "num_person": num_person,
        "visible_keypoints": visible_count,
        "person_conf": round(person_conf, 4),
        "image_out": str(image_out_path),
        "label_out": str(label_out_path),
        "preview_out": str(preview_out_path)
    })


# =========================
# 保存 summary CSV
# =========================
with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "class",
            "image",
            "status",
            "num_person",
            "visible_keypoints",
            "person_conf",
            "image_out",
            "label_out",
            "preview_out"
        ]
    )
    writer.writeheader()
    writer.writerows(rows)


# =========================
# 写 data.yaml
# =========================
dataset_path = str(OUT_ROOT).replace("\\", "/")

data_yaml = OUT_ROOT / "data.yaml"
data_yaml.write_text(
f"""path: "{dataset_path}"
train: images/train
val: images/train

kpt_shape: [17, 3]
flip_idx: [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]

names:
  0: person
""",
    encoding="utf-8"
)


print("\nDone.")
print("Output dataset:", OUT_ROOT)
print("Summary CSV:", CSV_PATH)
print("data.yaml:", data_yaml)

print("\nStatus summary:")
for k, v in sorted(counter.items()):
    print(f"{k:24s}: {v}")

In [ ]:
from pathlib import Path
import csv
import shutil
from collections import Counter

BASE_DIR = Path(r"D:/Documents/2026-04/Software/yoga train")

DATASET_ROOT = BASE_DIR / "pseudo_yolo_pose_dataset"
CSV_PATH = DATASET_ROOT / "pseudo_label_summary.csv"

REVIEW_ROOT = BASE_DIR / "pseudo_review_workspace"
REJECT_DIR = REVIEW_ROOT / "reject"

# 清空旧审核目录，避免混乱
if REVIEW_ROOT.exists():
    shutil.rmtree(REVIEW_ROOT)

REJECT_DIR.mkdir(parents=True, exist_ok=True)

status_counter = Counter()
skipped_counter = Counter()

with open(CSV_PATH, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

for row in rows:
    cls = row["class"]
    status = row["status"]
    preview_out = row.get("preview_out", "").strip()

    # 跳过没有 preview 路径的记录
    if not preview_out:
        skipped_counter["empty_preview_path"] += 1
        continue

    preview_path = Path(preview_out)

    # 跳过不存在的路径
    if not preview_path.exists():
        skipped_counter["preview_not_exists"] += 1
        continue

    # 跳过文件夹，避免 Path("") -> "." 这类问题
    if not preview_path.is_file():
        skipped_counter["preview_not_file"] += 1
        continue

    dst_dir = REVIEW_ROOT / status / cls
    dst_dir.mkdir(parents=True, exist_ok=True)

    dst = dst_dir / preview_path.name
    shutil.copy2(preview_path, dst)

    status_counter[status] += 1

print("Review workspace created:", REVIEW_ROOT)

print("\nCopied preview status summary:")
for status, count in sorted(status_counter.items()):
    print(f"{status:24s}: {count}")

print("\nSkipped summary:")
for reason, count in sorted(skipped_counter.items()):
    print(f"{reason:24s}: {count}")

print("\nReject folder:")
print(REJECT_DIR)

remove all reject data

In [ ]:
from pathlib import Path
import csv
import shutil
from collections import Counter

BASE_DIR = Path(r"D:/Documents/2026-04/Software/yoga train")

DATASET_ROOT = BASE_DIR / "pseudo_yolo_pose_dataset"
CSV_PATH = DATASET_ROOT / "pseudo_label_summary.csv"

REVIEW_ROOT = BASE_DIR / "pseudo_review_workspace"
REJECT_DIR = REVIEW_ROOT / "reject"

REMOVED_ROOT = DATASET_ROOT / "removed_by_review"
REMOVED_IMG_DIR = REMOVED_ROOT / "images"
REMOVED_LABEL_DIR = REMOVED_ROOT / "labels"
REMOVED_PREVIEW_DIR = REMOVED_ROOT / "previews"

for p in [REMOVED_IMG_DIR, REMOVED_LABEL_DIR, REMOVED_PREVIEW_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# 读取 reject 文件名
reject_files = []
for ext in ["*.jpg", "*.jpeg", "*.png"]:
    reject_files.extend(list(REJECT_DIR.glob(ext)))

reject_names = {p.name for p in reject_files}

print("Reject preview files:", len(reject_names))

if len(reject_names) == 0:
    print("Warning: reject folder is empty.")

# 读取 summary，建立 preview 文件名 -> image / label / preview 的映射
with open(CSV_PATH, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

preview_map = {}

for row in rows:
    preview_out = row.get("preview_out", "").strip()
    if not preview_out:
        continue

    preview_name = Path(preview_out).name
    preview_map[preview_name] = row

removed_counter = Counter()
not_found = []

for reject_name in sorted(reject_names):
    row = preview_map.get(reject_name)

    if row is None:
        not_found.append(reject_name)
        continue

    image_path = Path(row["image_out"]) if row.get("image_out") else None
    label_path = Path(row["label_out"]) if row.get("label_out") else None
    preview_path = Path(row["preview_out"]) if row.get("preview_out") else None

    # 移动训练图片
    if image_path and image_path.exists() and image_path.is_file():
        dst = REMOVED_IMG_DIR / image_path.name
        if dst.exists():
            dst.unlink()
        shutil.move(str(image_path), str(dst))
        removed_counter["images"] += 1

    # 移动标签
    if label_path and label_path.exists() and label_path.is_file():
        dst = REMOVED_LABEL_DIR / label_path.name
        if dst.exists():
            dst.unlink()
        shutil.move(str(label_path), str(dst))
        removed_counter["labels"] += 1

    # 移动原始 preview，不是 review workspace 里的副本
    if preview_path and preview_path.exists() and preview_path.is_file():
        dst = REMOVED_PREVIEW_DIR / preview_path.name
        if dst.exists():
            dst.unlink()
        shutil.move(str(preview_path), str(dst))
        removed_counter["previews"] += 1

print("\nRemoved files:")
for k, v in removed_counter.items():
    print(f"{k}: {v}")

print("\nReject previews not found in CSV:", len(not_found))
if not_found[:10]:
    print("Examples:", not_found[:10])

print("\nRemoved backup folder:")
print(REMOVED_ROOT)

In [ ]:
from pathlib import Path
from collections import Counter

BASE_DIR = Path(r"D:/Documents/2026-04/Software/yoga train")
DATASET_ROOT = BASE_DIR / "pseudo_yolo_pose_dataset"

IMG_DIR = DATASET_ROOT / "images" / "train"
LABEL_DIR = DATASET_ROOT / "labels" / "train"

image_files = []
for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]:
    image_files.extend(list(IMG_DIR.glob(ext)))

label_files = list(LABEL_DIR.glob("*.txt"))

print("Remaining images:", len(image_files))
print("Remaining labels:", len(label_files))

# 检查图片和标签是否一一对应
image_stems = {p.stem for p in image_files}
label_stems = {p.stem for p in label_files}

missing_labels = image_stems - label_stems
missing_images = label_stems - image_stems

print("Images without labels:", len(missing_labels))
print("Labels without images:", len(missing_images))

if missing_labels:
    print("Example missing labels:", list(missing_labels)[:10])

if missing_images:
    print("Example missing images:", list(missing_images)[:10])

# 检查 label 长度
length_counter = Counter()
empty_labels = 0

for lf in label_files:
    text = lf.read_text(encoding="utf-8").strip()

    if not text:
        empty_labels += 1
        continue

    lines = text.splitlines()
    for line in lines:
        parts = line.strip().split()
        length_counter[len(parts)] += 1

print("\nEmpty label files:", empty_labels)
print("Label value length statistics:")
for length, count in sorted(length_counter.items()):
    print(f"length={length}, count={count}")

print("\nExpected YOLO pose length for 17 keypoints:")
print("1 class + 4 bbox + 17*3 keypoints = 56")

split dataset in train/val

In [ ]:
from pathlib import Path
import shutil
import random
from collections import defaultdict, Counter

BASE_DIR = Path(r"D:/Documents/2026-04/Software/yoga train")

SRC_ROOT = BASE_DIR / "pseudo_yolo_pose_dataset"
SRC_IMG_DIR = SRC_ROOT / "images" / "train"
SRC_LABEL_DIR = SRC_ROOT / "labels" / "train"

OUT_ROOT = BASE_DIR / "pseudo_yolo_pose_dataset_split"

# 如果之前运行过，清空旧 split
if OUT_ROOT.exists():
    shutil.rmtree(OUT_ROOT)

TRAIN_IMG_DIR = OUT_ROOT / "images" / "train"
VAL_IMG_DIR = OUT_ROOT / "images" / "val"
TRAIN_LABEL_DIR = OUT_ROOT / "labels" / "train"
VAL_LABEL_DIR = OUT_ROOT / "labels" / "val"

for p in [TRAIN_IMG_DIR, VAL_IMG_DIR, TRAIN_LABEL_DIR, VAL_LABEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

VAL_RATIO = 0.2
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# 收集图片
image_files = []
for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]:
    image_files.extend(list(SRC_IMG_DIR.glob(ext)))

image_files = sorted(image_files)

print("Total source images:", len(image_files))

# 根据文件名前缀分组，例如 Downdog_xxx, Plank_xxx
groups = defaultdict(list)

for img_path in image_files:
    cls_name = img_path.stem.split("_")[0]
    groups[cls_name].append(img_path)

print("\nSource class distribution:")
for cls, files in sorted(groups.items()):
    print(f"{cls}: {len(files)}")

train_count = Counter()
val_count = Counter()

for cls, files in groups.items():
    files = files.copy()
    random.shuffle(files)

    val_num = max(1, int(len(files) * VAL_RATIO))
    val_files = files[:val_num]
    train_files = files[val_num:]

    for split_name, split_files in [("train", train_files), ("val", val_files)]:
        for img_path in split_files:
            label_path = SRC_LABEL_DIR / f"{img_path.stem}.txt"

            if not label_path.exists():
                print("Missing label:", label_path)
                continue

            if split_name == "train":
                dst_img = TRAIN_IMG_DIR / img_path.name
                dst_label = TRAIN_LABEL_DIR / label_path.name
                train_count[cls] += 1
            else:
                dst_img = VAL_IMG_DIR / img_path.name
                dst_label = VAL_LABEL_DIR / label_path.name
                val_count[cls] += 1

            shutil.copy2(img_path, dst_img)
            shutil.copy2(label_path, dst_label)

# 写 data.yaml
dataset_path = str(OUT_ROOT).replace("\\", "/")

data_yaml = OUT_ROOT / "data.yaml"
data_yaml.write_text(
f"""path: "{dataset_path}"
train: images/train
val: images/val

kpt_shape: [17, 3]
flip_idx: [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]

names:
  0: person
""",
    encoding="utf-8"
)

print("\nTrain distribution:")
for cls, count in sorted(train_count.items()):
    print(f"{cls}: {count}")

print("\nVal distribution:")
for cls, count in sorted(val_count.items()):
    print(f"{cls}: {count}")

print("\nOutput split dataset:", OUT_ROOT)
print("data.yaml:", data_yaml)

In [ ]:
from pathlib import Path
from collections import Counter

BASE_DIR = Path(r"D:/Documents/2026-04/Software/yoga train")
DATASET_ROOT = BASE_DIR / "pseudo_yolo_pose_dataset_split"

for split in ["train", "val"]:
    img_dir = DATASET_ROOT / "images" / split
    label_dir = DATASET_ROOT / "labels" / split

    image_files = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]:
        image_files.extend(list(img_dir.glob(ext)))

    label_files = list(label_dir.glob("*.txt"))

    image_stems = {p.stem for p in image_files}
    label_stems = {p.stem for p in label_files}

    print("=" * 60)
    print(split)
    print("images:", len(image_files))
    print("labels:", len(label_files))
    print("images without labels:", len(image_stems - label_stems))
    print("labels without images:", len(label_stems - image_stems))

    length_counter = Counter()
    empty_labels = 0

    for lf in label_files:
        text = lf.read_text(encoding="utf-8").strip()
        if not text:
            empty_labels += 1
            continue

        for line in text.splitlines():
            length_counter[len(line.strip().split())] += 1

    print("empty labels:", empty_labels)
    print("label length stats:", dict(length_counter))

print("\ndata.yaml content:")
print((DATASET_ROOT / "data.yaml").read_text(encoding="utf-8"))

test for traning, only 5 epoch

In [ ]:
from ultralytics import YOLO
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model = YOLO("yolo11s-pose.pt")

results = model.train(
    data=r"D:/Documents/2026-04/Software/yoga train/pseudo_yolo_pose_dataset_split/data.yaml",
    epochs=5,
    imgsz=640,
    batch=8,
    device=0,
    workers=0,   # Windows + Notebook 环境建议先设为 0，避免多进程问题
    project=r"D:/Documents/2026-04/Software/yoga train/runs",
    name="yolo11s_pose_pseudo_sanity",
    exist_ok=True,
    pretrained=True,
    optimizer="auto",
    seed=42
)

In [ ]:
from ultralytics import YOLO
import os
from IPython.display import Image, display

# 微调 5 epoch 后的模型
model = YOLO(r"D:/Documents/2026-04/Software/yoga train/runs/yolo11s_pose_pseudo_sanity/weights/best.pt")

image_paths = [
    r"D:/Documents/2026-04/Software/yoga train/squat.jpg",
    r"D:/Documents/2026-04/Software/yoga train/Downward-Dog.jpg"
]

results = model.predict(
    source=image_paths,
    conf=0.25,
    imgsz=640,
    device=0,
    save=True,
    project=r"D:/Documents/2026-04/Software/yoga train/runs",
    name="test_finetuned_sanity_best",
    exist_ok=True
)

print("Prediction finished.")
print("Saved to:", results[0].save_dir)

save_dir = results[0].save_dir

for file in os.listdir(save_dir):
    if file.lower().endswith((".jpg", ".jpeg", ".png")):
        print(file)
        display(Image(filename=os.path.join(save_dir, file)))

train model with 50 epoch

In [ ]:
from ultralytics import YOLO
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model = YOLO("yolo11s-pose.pt")

results = model.train(
    data=r"D:/Documents/2026-04/Software/yoga train/pseudo_yolo_pose_dataset_split/data.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    device=0,
    workers=0,
    project=r"D:/Documents/2026-04/Software/yoga train/runs",
    name="yolo11s_pose_yoga5_singleperson_50ep",
    exist_ok=True,
    pretrained=True,
    optimizer="auto",
    seed=42,
    patience=15,
    plots=True
)

visable result

In [ ]:
from pathlib import Path
from ultralytics import YOLO
from IPython.display import Image, display
import random
import os

BASE_DIR = Path(r"D:/Documents/2026-04/Software/yoga train")

model_path = BASE_DIR / "runs" / "yolo11s_pose_yoga5_singleperson_50ep" / "weights" / "best.pt"
image_root = BASE_DIR / "relabel_images_all"

model = YOLO(str(model_path))

classes = ["Downdog", "Plank", "Tree", "Goddess", "Warrior2"]

# 每类抽几张测试
SAMPLE_PER_CLASS = 3

test_images = []

random.seed(123)

for cls in classes:
    cls_dir = image_root / cls

    imgs = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"]:
        imgs.extend(list(cls_dir.glob(ext)))

    imgs = sorted(imgs)

    if len(imgs) == 0:
        print(f"No images found for {cls}")
        continue

    sample_imgs = random.sample(imgs, min(SAMPLE_PER_CLASS, len(imgs)))
    test_images.extend(sample_imgs)

print("Total test images:", len(test_images))
for p in test_images:
    print(p)

results = model.predict(
    source=[str(p) for p in test_images],
    conf=0.5,
    imgsz=640,
    device=0,
    max_det=1,
    save=True,
    project=str(BASE_DIR / "runs"),
    name="test_yoga5_best_visual",
    exist_ok=True
)

print("Prediction finished.")
print("Saved to:", results[0].save_dir)

save_dir = results[0].save_dir

for file in os.listdir(save_dir):
    if file.lower().endswith((".jpg", ".jpeg", ".png")):
        print(file)
        display(Image(filename=os.path.join(save_dir, file)))